In [ ]:
import pandas as pd
import numpy as np


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [ ]:
!kaggle datasets download -d bravehart101/sample-supermarket-dataset
!unzip -o sample-supermarket-dataset.zip

Dataset URL: https://www.kaggle.com/datasets/bravehart101/sample-supermarket-dataset
License(s): CC0-1.0
100% 164k/164k [00:00<00:00, 104MB/s]

Archive:  sample-supermarket-dataset.zip
  inflating: SampleSuperstore.csv    


In [ ]:
df = pd.read_csv("sample-supermarket-dataset.zip")

In [ ]:
df

,Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub-Category,Sales,Quantity,Discount,Profit
0,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,261.9600,2,0.00,41.9136
1,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,731.9400,3,0.00,219.5820
2,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,14.6200,2,0.00,6.8714
3,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,957.5775,5,0.45,-383.0310
4,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,22.3680,2,0.20,2.5164
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9989,Second Class,Consumer,United States,Miami,Florida,33180,South,Furniture,Furnishings,25.2480,3,0.20,4.1028
9990,Standard Class,Consumer,United States,Costa Mesa,California,92627,West,Furniture,Furnishings,91.9600,2,0.00,15.6332
9991,Standard Class,Consumer,United States,Costa Mesa,California,92627,West,Technology,Phones,258.5760,2,0.20,19.3932
9992,Standard Class,Consumer,United States,Costa Mesa,California,92627,West,Office Supplies,Paper,29.6000,4,0.00,13.3200


In [ ]:
df.shape

(9994, 13)

In [ ]:
df.isnull().sum()

,0
Ship Mode,0
Segment,0
Country,0
City,0
State,0
Postal Code,0
Region,0
Category,0
Sub-Category,0
Sales,0


In [ ]:
df.columns.tolist()

['Ship Mode',
 'Segment',
 'Country',
 'City',
 'State',
 'Postal Code',
 'Region',
 'Category',
 'Sub-Category',
 'Sales',
 'Quantity',
 'Discount',
 'Profit']

In [ ]:
df["Profit_Margin_%"] = (df["Profit"] / df['Sales']) * 100

In [ ]:
top_sales_catigories = df.groupby("Category")[["Sales", "Profit"]].sum().sort_values(by="Sales", ascending=False)

In [ ]:
category_margin = df.groupby("Category")["Profit_Margin_%"].mean().to_frame()

In [ ]:
analysis_1 = top_sales_catigories.merge(category_margin, on="Category")
print("---Kategoryalar bo'yicha sotuv,foyda va marja tahlili ---")
print(analysis_1)
#

---Kategoryalar bo'yicha sotuv,foyda va marja tahlili ---
                       Sales       Profit  Profit_Margin_%
Category                                                  
Technology       836154.0330  145454.9481        15.613805
Furniture        741999.7953   18451.2728         3.878353
Office Supplies  719047.0320  122490.8008        13.803029


In [ ]:
furinture_df = df[df['Category'] == "Furniture"]

In [ ]:
furinture_analysis = furinture_df.groupby("Sub-Category").agg(Total_Sales=('Sales', 'sum'), Total_Profit=("Profit", "sum"), Avg_Discount=("Discount", "mean")).sort_values(by='Total_Profit')

In [ ]:
furinture_analysis['Margin_%'] = (furinture_analysis["Total_Profit"]/ furinture_analysis["Total_Sales"])*100
print("--- Furiniture Ichidagi Sub-Categoryalar Tahlili ---")
print(furinture_analysis)

--- Furiniture Ichidagi Sub-Categoryalar Tahlili ---
              Total_Sales  Total_Profit  Avg_Discount   Margin_%
Sub-Category                                                    
Tables        206965.5320   -17725.4811      0.261285  -8.564460
Bookcases     114879.9963    -3472.5560      0.211140  -3.022768
Furnishings    91705.1640    13059.1436      0.138349  14.240358
Chairs        328449.1030    26590.1663      0.170178   8.095673


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Grafik uchun ma'lumotlarni tayyorlash (Zarar bo'yicha tartiblaymiz)
sub_cats = furinture_analysis.index.tolist()
profit = furinture_analysis['Total_Profit'].tolist()
discount = (furinture_analysis['Avg_Discount'] * 100).tolist() # Foizga o'tkazamiz

# 1 ta qator va 2 ta ustundan iborat grafik maydoni yaratamiz
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Sub-Kategoriyalar bo'yicha Sof Foyda ($)", "O'rtacha Chegirma stavkasi (%)")
)

# 1-Grafik: Sof foyda (Bar chart)
fig.add_trace(
    go.Bar(
        x=sub_cats, y=profit,
        marker_color=['red' if x < 0 else 'green' for x in profit],
        name="Sof Foyda"
    ),
    row=1, col=1
)

# 2-Grafik: Chegirmalar (Bar chart)
fig.add_trace(
    go.Bar(
        x=sub_cats, y=discount,
        marker_color='blue',
        name="Chegirma %"
    ),
    row=1, col=2
)

# Dashboard dizaynini sozlash
fig.update_layout(
    title_text="Furniture (Mebel) Kategoriyasi Ichki Analitik Dashbordi",
    showlegend=False,
    template="plotly_white",
    height=500
)

# Grafikni ko'rsatish
fig.show()